## 3. The 8 Quantitative Feature Clusters (Summary Reference)

All 33 predictive features are computed **strictly from historical data up to $T-1$ Close** (or completed prior sessions) with **zero data leakage**.

> Detailed formulas, boundary constraints, and null-coalescing defaults are documented in [`src/mdk_trading_oracle/models/day_start/FEATURES.md`](../src/mdk_trading_oracle/models/day_start/FEATURES.md).

| Cluster | Feature Name | Type & Lag | Mathematical Formulation | Microstructure & Behavioral Rationale |
| :--- | :--- | :--- | :--- | :--- |
| **C1: Closing Momentum** | `feat_bofa_w4_net_flow_tl` | Flow ($T-1$) | $\text{Buy}_{\text{MLB}, \text{W4}} - \text{Sell}_{\text{MLB}, \text{W4}}$ | Unfinished MOC / VWAP benchmark parent orders carry over into next morning open. |
| **C1: Closing Momentum** | `feat_bofa_w4_turnover_tl` | Flow ($T-1$) | $\text{Buy}_{\text{MLB}, \text{W4}} + \text{Sell}_{\text{MLB}, \text{W4}}$ | Higher closing volume confirms conviction behind the directional move. |
| **C1: Closing Momentum** | `feat_w4_flow_acceleration_ratio` | Ratio ($T-1$) | $\frac{\text{Net Flow}_{\text{MLB}, \text{W4}}}{\|\text{Net Flow}_{\text{MLB}, \text{Day}}\| + \epsilon}$ | Values $> 0.5$ indicate aggressive end-of-day parent order urgency. |
| **C2: Inventory Saturation** | `feat_bofa_cum_net_flow_5d_tl` | Rolling ($T-1$) | $\sum_{k=1}^5 \text{Net Flow}_{\text{MLB}, T-k}$ | Algorithmic inventory ceilings; multi-day accumulation triggers rebalancing. |
| **C2: Inventory Saturation** | `feat_top5_cum_net_flow_5d_tl` | Rolling ($T-1$) | $\sum_{k=1}^5 \sum_{b \in \text{Top5}} \text{Net Flow}_{b, T-k}$ | Cumulative 5-day positioning of domestic powerhouses (`IYM`, `YKR`, `AKM`, etc.). |
| **C2: Inventory Saturation** | `feat_bofa_flow_zscore_20d` | Z-Score ($T-1$) | $\frac{\text{Flow}_{\text{MLB}, T-1} - \mu_{20d}}{\sigma_{20d} + \epsilon}$ | $|Z| > 2.0$ represents statistical tail events (extreme institutional pressure). |
| **C2: Inventory Saturation** | `feat_bofa_*_flow_prev_day` | Sector Flow ($T-1$) | $\text{Net Flow}_{\text{MLB}, \text{Sector}, T-1}$ | BofA previous-day flow across Banking, Transportation, Holding, Energy, Defense. |
| **C3: Cost Basis & PnL** | `feat_bofa_cost_basis_spread_20d_pct` | Spread ($T-1$) | $\frac{\bar{P}_{\text{Close}} - \text{VWAP}_{\text{Buy}, 20d}}{\text{VWAP}_{\text{Buy}, 20d}} \times 100$ | Spread $> +5\%$ prompts liquidity fades; $< -4\%$ prompts defense buying. |
| **C3: Cost Basis & PnL** | `feat_prev_day_close_vs_vwap_spread_pct` | Spread ($T-1$) | $\frac{\bar{P}_{\text{Close}} - \text{VWAP}_{\text{Market}}}{\text{VWAP}_{\text{Market}}} \times 100$ | Previous-day closing price premium/discount relative to intraday market VWAP. |
| **C4: Competitor Imbalance** | `feat_top5_domestic_w4_net_flow_tl` | Flow ($T-1$) | $\sum_{b \in \text{Top5}} \text{Net Flow}_{b, \text{W4}, T-1}$ | Closing posture of top 5 domestic brokerages. |
| **C4: Competitor Imbalance** | `feat_bofa_vs_top5_w4_flow_delta_tl` | Divergence ($T-1$) | $\text{Flow}_{\text{MLB}, \text{W4}} - \text{Flow}_{\text{Top5}, \text{W4}}$ | Divergence $> +30\text{M TL}$ signals foreign flow overpowering domestic resistance. |
| **C5: Hegemony & Concentration** | `feat_bofa_prev_day_market_share` | Ratio ($T-1$) | $\frac{\text{Turnover}_{\text{MLB}}}{\text{Turnover}_{\text{Market}}}$ | BofA exchange turnover market share (quantifies institutional pricing power). |
| **C5: Hegemony & Concentration** | `feat_institutional_hegemony_share` | Ratio ($T-1$) | $\frac{\text{Turnover}_{\text{MLB}} + \text{Turnover}_{\text{Top5}}}{\text{Turnover}_{\text{Market}}}$ | Combined institutional turnover concentration of BofA + Top 5 domestic brokers. |
| **C6: Volatility & Stress** | `feat_market_avg_return_pct` | Return ($T-1$) | $\frac{1}{N}\sum_{s=1}^N \text{Return}_{s, T-1}$ | Market-wide return baseline across tracked liquid equities. |
| **C6: Volatility & Stress** | `feat_market_avg_range_pct` | Volatility ($T-1$) | $\frac{1}{N}\sum_{s=1}^N \frac{P_{\text{High}} - P_{\text{Low}}}{P_{\text{Low}}} \times 100$ | Cross-sectional high-low price range (intraday market volatility proxy). |
| **C7: Calendar Dynamics** | `day_of_week`, `is_monday`, `is_friday` | Calendar ($T$) | Integer / Boolean | Monday opening re-allocation vs Friday delta hedging / risk reduction. |
| **C8: Macro Rates & Shock** | `feat_macro_interest_rate` | Level ($T-1$) | $\text{Rate}_{T-1}$ (TCMB 1-Week Repo %) | Systemic cost of capital and equity hurdle rate. |
| **C8: Macro Rates & Shock** | `feat_macro_rate_shock_decay` | Impulse ($T-1$) | $\frac{\Delta \text{Rate}_{\text{bps}}}{\max(1, \Delta \text{days})}$ | Decay-weighted rate shock ($100\%$ on $T=0,1$, decaying at $1/d$ on $T \ge 2$). |
| **C8: Macro Rates & Shock** | `feat_macro_rate_spread_vs_30d_mean` | Delta ($T-1$) | $(\text{Rate}_{T-1} - \overline{\text{Rate}}_{30d}) \times 100$ | Monetary policy trend vs 30-day moving average (tightening/easing). |
| **C8: Macro Rates & Shock** | `feat_macro_daily_carry_cost_bps` | Level ($T-1$) | $\frac{\text{Rate}_{T-1}}{365} \times 100\text{ (bps)}$ | Overnight financing carry cost of holding long equity inventory. |


In [ ]:
import duckdb
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

from mdk_trading_oracle.core.db import DuckDBManager
from mdk_trading_oracle.core.config import get_settings
from mdk_trading_oracle.models.day_start import (
    DayStartFeatureExtractor,
    DayStartForecaster,
    DayStartModelArena,
    DayStartNaivePersistenceModel,
    DayStartRollingMeanModel,
    DayStartBayesianModel,
    DayStartPyMCModel,
    DayStartLightGBMModel,
    DayStartXGBoostModel,
)

settings = get_settings()
print(f"[OK] DuckDB Database: {settings.duckdb_path}")
print(f"[OK] Data Directory: {settings.data_dir}")


## 2. Feature Extraction: Assembling the 8 Feature Clusters

We extract features computed strictly at $T-1$ Close from our 6 Silver fact tables with **zero data leakage**.


In [ ]:
db = DuckDBManager(read_only=True)
extractor = DayStartFeatureExtractor(db, target_broker_id="MLB")
df_pl = extractor.extract_features()
df = df_pl.to_pandas()

print(f"[OK] Extracted {len(df)} historical trading sessions with {len(df.columns)} features.")
display(df.head(5)[["trade_date", "day_of_week", "is_monday", "feat_bofa_w4_net_flow_tl", 
                   "feat_bofa_vs_top5_w4_flow_delta_tl", "feat_bofa_cost_basis_spread_20d_pct", 
                   "target_open_net_flow_tl", "target_open_direction"]])


## 3. Feature Importance & Correlation Analysis

How do yesterday's closing signals, competitor imbalances, and cost basis spreads correlate with today's opening net flow?


In [ ]:
# Calculate correlations with the target opening net flow
feat_cols = [c for c in df.columns if c.startswith("feat_") or c in ["is_monday", "is_friday"]]
corrs = df[feat_cols + ["target_open_net_flow_tl"]].corr()["target_open_net_flow_tl"].drop("target_open_net_flow_tl").sort_values()

fig_corr = px.bar(
    x=corrs.values,
    y=corrs.index,
    orientation="h",
    title="Feature Correlations with Day-Start Opening Net Flow (Window 1)",
    labels={"x": "Pearson Correlation", "y": "Feature Name"},
    color=corrs.values,
    color_continuous_scale="RdBu_r",
    height=600
)
fig_corr.update_layout(template="plotly_dark", showlegend=False)
fig_corr.show()


## 4. Multi-Model Arena & Auto-Champion Tournament (Walk-Forward Validation)

We run an expanding-window **Walk-Forward Validation Tournament** across all 6 candidate paradigms (Baselines, LightGBM, XGBoost, Bayesian Ridge, PyMC GLM).
Models are trained strictly on past trading sessions ($1 \dots t-1$) to forecast session $t$, guaranteeing **zero lookahead bias**.


In [ ]:
X = df.drop(columns=["target_open_net_flow_tl", "target_open_direction"], errors="ignore")
y = df["target_open_net_flow_tl"]

# Run Automated Walk-Forward Tournament across all 6 candidates
arena = DayStartModelArena()
scoreboard_df, champion_model = arena.run_tournament(X, y, min_train_samples=5, eval_window_days=20)

champion_name = scoreboard_df.iloc[0]["Model"]
champ_hit_rate = scoreboard_df.iloc[0]["hit_rate_pct"]
champ_picp = scoreboard_df.iloc[0]["picp_90_pct"]
champ_rmse = scoreboard_df.iloc[0]["rmse_million_tl"]

display(HTML(f"""
<div style="background: linear-gradient(135deg, #1b4332 0%, #081c15 100%); padding: 18px 24px; border-radius: 12px; border-left: 6px solid #52b788; margin-bottom: 20px; color: #fff; box-shadow: 0 4px 15px rgba(0,0,0,0.3);">
    <h3 style="margin: 0; color: #52b788;">Champion Crowned by Auto-Arena: {champion_name}</h3>
    <p style="margin: 6px 0 0 0; font-size: 14px; opacity: 0.95;">
        <b>Out-of-Sample Hit Rate:</b> <span style="color: #74c69d; font-weight: bold;">{champ_hit_rate:.1f}%</span> &nbsp;|&nbsp; 
        <b>90% Credible Interval Coverage (PICP):</b> <span style="color: #74c69d; font-weight: bold;">{champ_picp:.1f}%</span> &nbsp;|&nbsp; 
        <b>RMSE:</b> {champ_rmse:.2f}M TL
    </p>
</div>
"""))

display(HTML("<h3>Out-of-Sample Walk-Forward Scoreboard</h3>"))
display(scoreboard_df.style.highlight_max(subset=["hit_rate_pct", "picp_90_pct"], color="#1b4332")
                           .highlight_min(subset=["mae_million_tl", "rmse_million_tl"], color="#1b4332"))


## 5. Live Next-Day Forecast (Actionable Trading Signal for Tomorrow)

Using the dynamically crowned champion model from the Tournament, we extract features strictly from the latest market close and generate the **live forecast for the upcoming morning opening auction**.


In [ ]:
# Initialize Forecaster with the dynamically crowned champion
forecaster = DayStartForecaster(db, model_type=champion_model.model_name)
live_forecast = forecaster.forecast_next_day()

# Display Live Signal Card for Traders
next_date = live_forecast.forecast_date
pred_flow = live_forecast.predicted_net_flow_tl / 1e6
lower_90 = live_forecast.predicted_flow_lower_90 / 1e6
upper_90 = live_forecast.predicted_flow_upper_90 / 1e6
direction = live_forecast.predicted_direction
confidence = live_forecast.direction_confidence * 100
playbook = live_forecast.predicted_playbook
buy_sec = live_forecast.top_predicted_buy_sector
sell_sec = live_forecast.top_predicted_sell_sector

dir_color = "#06d6a0" if "ACCUMULATE" in direction else ("#ef476f" if "DISTRIBUTE" in direction else "#ffd166")

display(HTML(f"""
<div style="background: linear-gradient(135deg, #0d1b2a 0%, #1b263b 100%); padding: 24px; border-radius: 14px; border: 2px solid {dir_color}; box-shadow: 0 8px 25px rgba(0,0,0,0.4); margin-bottom: 25px; color: #fff;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid rgba(255,255,255,0.15); padding-bottom: 12px; margin-bottom: 16px;">
        <h2 style="margin: 0; color: #e0e1dd; font-size: 20px;">Live Forecast for Upcoming Session: <span style="color: #00b4d8;">{next_date}</span></h2>
        <span style="background: {dir_color}; color: #000; font-weight: bold; padding: 6px 14px; border-radius: 20px; font-size: 14px;">{direction} ({confidence:.1f}% Conviction)</span>
    </div>
    <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 16px; font-size: 15px;">
        <div style="background: rgba(255,255,255,0.05); padding: 14px; border-radius: 8px;">
            <div style="color: #778da9; font-size: 13px; text-transform: uppercase;">Expected Opening Net Flow</div>
            <div style="font-size: 22px; font-weight: bold; color: {dir_color}; margin-top: 4px;">{pred_flow:+,.2f} M TL</div>
            <div style="color: #778da9; font-size: 12px; margin-top: 2px;">90% CI: [{lower_90:+,.1f}M, {upper_90:+,.1f}M]</div>
        </div>
        <div style="background: rgba(255,255,255,0.05); padding: 14px; border-radius: 8px;">
            <div style="color: #778da9; font-size: 13px; text-transform: uppercase;">Institutional Playbook</div>
            <div style="font-size: 20px; font-weight: bold; color: #00b4d8; margin-top: 4px;">{playbook}</div>
            <div style="color: #778da9; font-size: 12px; margin-top: 2px;">Champion Model: {champion_name}</div>
        </div>
        <div style="background: rgba(255,255,255,0.05); padding: 14px; border-radius: 8px;">
            <div style="color: #778da9; font-size: 13px; text-transform: uppercase;">Top Sector Rotation Focus</div>
            <div style="font-size: 16px; margin-top: 4px;"><b>Buy:</b> <span style="color: #52b788;">{buy_sec}</span></div>
            <div style="font-size: 16px; margin-top: 2px;"><b>Sell:</b> <span style="color: #e63946;">{sell_sec}</span></div>
        </div>
    </div>
</div>
"""))


## 6. Historical Backtest Track Record: Predicted vs Actual Opening Net Flow & 90% Confidence Ribbon

Visualizing historical out-of-sample and backtested performance of the crowned Champion model against actual opening net flows.


In [ ]:
# Generate historical backtest track record using crowned champion
backtest_forecasts = forecaster.backtest_all_history()

predictions = []
lowers = []
uppers = []
playbooks = []
directions = []
confidences = []

for res in backtest_forecasts:
    predictions.append(res.predicted_net_flow_tl / 1e6)
    lowers.append(res.predicted_flow_lower_90 / 1e6)
    uppers.append(res.predicted_flow_upper_90 / 1e6)
    playbooks.append(res.predicted_playbook)
    directions.append(res.predicted_direction)
    confidences.append(res.direction_confidence)

chart_df = df.copy()
chart_df["trade_date"] = chart_df["trade_date"].astype(str).str.slice(0, 10)
chart_df["pred_flow_m"] = predictions
chart_df["lower_90_m"] = lowers
chart_df["upper_90_m"] = uppers
chart_df["actual_flow_m"] = df["target_open_net_flow_tl"] / 1e6
chart_df["playbook"] = playbooks
chart_df["pred_direction"] = directions
chart_df["confidence"] = confidences

fig = go.Figure()

# 90% Confidence Interval Shaded Band
fig.add_trace(go.Scatter(
    x=chart_df["trade_date"].tolist() + chart_df["trade_date"].tolist()[::-1],
    y=chart_df["upper_90_m"].tolist() + chart_df["lower_90_m"].tolist()[::-1],
    fill="toself",
    fillcolor="rgba(0, 180, 216, 0.15)",
    line=dict(color="rgba(255,255,255,0)"),
    name="90% Credible Interval",
    hoverinfo="skip"
))

# Predicted Flow
fig.add_trace(go.Scatter(
    x=chart_df["trade_date"],
    y=chart_df["pred_flow_m"],
    mode="lines+markers",
    name="Predicted Opening Net Flow (TL M)",
    line=dict(color="#00b4d8", width=3),
    marker=dict(size=8, symbol="diamond")
))

# Actual Flow
fig.add_trace(go.Scatter(
    x=chart_df["trade_date"],
    y=chart_df["actual_flow_m"],
    mode="lines+markers",
    name="Actual Window 1 Net Flow (TL M)",
    line=dict(color="#ffb703", width=2, dash="dash"),
    marker=dict(size=7, symbol="circle")
))

fig.update_layout(
    title="Bank of America Day-Start Backtest: Predicted vs Actual Opening Net Flow (Million TL)",
    xaxis_title="Trading Date",
    yaxis_title="Net Flow (Million TL)",
    template="plotly_dark",
    hovermode="x unified",
    height=550
)
fig.show()


## 7. Interactive Historical Session Inspector & Playbook Breakdown

Select any historical date to inspect the model's opening conviction, competitor closing posture, and sector allocation forecast.


In [ ]:
date_options = chart_df["trade_date"].tolist()

date_dropdown = widgets.Dropdown(
    options=date_options,
    value=date_options[-1] if date_options else None,
    description="Date:",
    style={"description_width": "initial"}
)

output = widgets.Output()

def update_session(change):
    with output:
        output.clear_output()
        sel_date = change["new"]
        if not sel_date:
            return
        matches = chart_df[chart_df["trade_date"] == str(sel_date)]
        if len(matches) == 0:
            display(HTML(f"<p style='color: orange;'>No data for {sel_date}</p>"))
            return
        row = matches.iloc[0]
        
        dir_color = "#06d6a0" if "ACCUMULATE" in row["pred_direction"] else ("#ef476f" if "DISTRIBUTE" in row["pred_direction"] else "#ffd166")
        
        html_card = f"""
        <div style="background: #1e1e2e; padding: 20px; border-radius: 12px; border-left: 6px solid {dir_color}; margin-bottom: 15px; color: #fff;">
            <h3 style="margin-top: 0;">Trading Session: {sel_date} ({'Monday (Rebalancing)' if row['is_monday'] else 'Regular Session'})</h3>
            <table style="width: 100%; border-collapse: collapse; font-size: 15px;">
                <tr>
                    <td><b>Predicted Direction:</b> <span style="color: {dir_color}; font-weight: bold;">{row['pred_direction']}</span></td>
                    <td><b>Conviction / Confidence:</b> {row['confidence'] * 100:.1f}%</td>
                </tr>
                <tr>
                    <td><b>Forecasted Net Flow:</b> {row['pred_flow_m']:+,.2f} M TL</td>
                    <td><b>90% Credible Range:</b> [{row['lower_90_m']:+,.2f} M, {row['upper_90_m']:+,.2f} M]</td>
                </tr>
                <tr>
                    <td><b>Institutional Playbook:</b> <span style="color: #00b4d8; font-weight: bold;">{row['playbook']}</span></td>
                    <td><b>Actual Window 1 Flow:</b> {row['actual_flow_m']:+,.2f} M TL</td>
                </tr>
                <tr>
                    <td><b>Yesterday W4 Net Flow:</b> {row['feat_bofa_w4_net_flow_tl'] / 1e6:+,.2f} M TL</td>
                    <td><b>Top-5 Competitor Closing Delta:</b> {row['feat_bofa_vs_top5_w4_flow_delta_tl'] / 1e6:+,.2f} M TL</td>
                </tr>
            </table>
        </div>
        """
        display(HTML(html_card))

date_dropdown.observe(update_session, names="value")
display(date_dropdown, output)
if date_dropdown.value:
    update_session({"new": date_dropdown.value})


## 8. Gold Tables & Production Performance Ledgers in DuckDB

Verifying the persisted production tables in DuckDB:
1. `gold_bofa_day_start_forecasts`: Pure upcoming live forecasts ($T+1$) strictly for tomorrow's market open.
2. `gold_bofa_day_start_performance`: Permanent historical performance tracking ledger recording prior forecasts matched against realized actual Window 1 market data.
3. `gold_bofa_day_start_backtests`: Dedicated historical walk-forward backtest simulation ledger.


In [ ]:
conn = db.get_connection()

print("1. Live Active Upcoming Forecast (gold_bofa_day_start_forecasts) - Strictly T+1:")
gold_forecasts_df = conn.execute("""
    SELECT 
        forecast_date,
        day_of_week,
        is_monday,
        predicted_open_net_flow_tl / 1e6 AS pred_net_flow_m_tl,
        predicted_direction,
        direction_confidence,
        predicted_playbook,
        top_predicted_buy_sector,
        top_predicted_sell_sector,
        model_name
    FROM gold_bofa_day_start_forecasts
    ORDER BY forecast_date DESC;
""").df()
display(gold_forecasts_df)

print("2. Historical Performance Tracking Ledger (gold_bofa_day_start_performance) - Latest 5 Sessions:")
gold_perf_df = conn.execute("""
    SELECT 
        trade_date,
        predicted_open_net_flow_tl / 1e6 AS pred_m_tl,
        actual_open_net_flow_tl / 1e6 AS actual_m_tl,
        error_open_net_flow_tl / 1e6 AS error_m_tl,
        absolute_error_tl / 1e6 AS abs_error_m_tl,
        predicted_direction,
        actual_direction,
        is_direction_hit,
        is_inside_90_ci,
        predicted_playbook
    FROM gold_bofa_day_start_performance
    ORDER BY trade_date DESC
    LIMIT 5;
""").df()
display(gold_perf_df)


### Historical Backtest Performance Dashboard (`gold_bofa_day_start_backtests`)

Calculating executive summary KPIs directly from the DuckDB backtest ledger:


In [ ]:
# Summary KPIs from DuckDB
backtest_kpis = conn.execute("""
    SELECT 
        COUNT(*) AS total_sessions,
        ROUND(AVG(CASE WHEN is_direction_hit THEN 1.0 ELSE 0.0 END) * 100, 1) AS hit_rate_pct,
        ROUND(AVG(CASE WHEN is_inside_90_ci THEN 1.0 ELSE 0.0 END) * 100, 1) AS picp_90_pct,
        ROUND(AVG(ABS(error_open_net_flow_tl)) / 1e6, 2) AS mae_m_tl,
        ROUND(SQRT(AVG(POWER(error_open_net_flow_tl, 2))) / 1e6, 2) AS rmse_m_tl
    FROM gold_bofa_day_start_backtests;
""").df().iloc[0]


kpi_html = f"""
<div style="display: flex; gap: 15px; margin-bottom: 20px;">
    <div style="flex: 1; background: #1e1e2e; padding: 15px; border-radius: 10px; border-left: 5px solid #00b4d8; color: #fff;">
        <div style="font-size: 12px; color: #888; text-transform: uppercase;">Total Evaluated Sessions</div>
        <div style="font-size: 24px; font-weight: bold; margin-top: 5px;">{int(backtest_kpis['total_sessions'])}</div>
    </div>
    <div style="flex: 1; background: #1e1e2e; padding: 15px; border-radius: 10px; border-left: 5px solid #06d6a0; color: #fff;">
        <div style="font-size: 12px; color: #888; text-transform: uppercase;">Out-of-Sample Hit Rate</div>
        <div style="font-size: 24px; font-weight: bold; margin-top: 5px; color: #06d6a0;">{backtest_kpis['hit_rate_pct']:.1f}%</div>
    </div>
    <div style="flex: 1; background: #1e1e2e; padding: 15px; border-radius: 10px; border-left: 5px solid #ffd166; color: #fff;">
        <div style="font-size: 12px; color: #888; text-transform: uppercase;">90% Credible Coverage (PICP)</div>
        <div style="font-size: 24px; font-weight: bold; margin-top: 5px; color: #ffd166;">{backtest_kpis['picp_90_pct']:.1f}%</div>
    </div>
    <div style="flex: 1; background: #1e1e2e; padding: 15px; border-radius: 10px; border-left: 5px solid #ef476f; color: #fff;">
        <div style="font-size: 12px; color: #888; text-transform: uppercase;">Mean Absolute Error (MAE)</div>
        <div style="font-size: 24px; font-weight: bold; margin-top: 5px;">{backtest_kpis['mae_m_tl']:,.2f} M TL</div>
    </div>
</div>
"""
display(HTML(kpi_html))


In [ ]:
print("2. Full Day-Start Backtest Ledger (gold_bofa_day_start_backtests):")
gold_backtests_full_df = conn.execute("""
    SELECT 
        trade_date,
        day_of_week,
        is_monday,
        predicted_open_net_flow_tl / 1e6 AS pred_net_flow_m,
        actual_open_net_flow_tl / 1e6 AS act_net_flow_m,
        error_open_net_flow_tl / 1e6 AS error_m,
        predicted_direction,
        actual_direction,
        is_direction_hit,
        is_inside_90_ci,
        direction_confidence,
        predicted_playbook,
        model_name
    FROM gold_bofa_day_start_backtests
    ORDER BY trade_date DESC;
""").df()

# Display formatted full backtest table
display(gold_backtests_full_df)


In [ ]:
print("3. Backtest Performance by Session Type (Day of Week):")
dow_perf_df = conn.execute("""
    SELECT 
        day_of_week,
        COUNT(*) AS sessions,
        ROUND(AVG(CASE WHEN is_direction_hit THEN 1.0 ELSE 0.0 END) * 100, 1) AS hit_rate_pct,
        ROUND(AVG(CASE WHEN is_inside_90_ci THEN 1.0 ELSE 0.0 END) * 100, 1) AS picp_90_pct,
        ROUND(AVG(ABS(error_open_net_flow_tl)) / 1e6, 2) AS mae_m_tl,
        ROUND(AVG(actual_open_net_flow_tl) / 1e6, 2) AS avg_actual_flow_m
    FROM gold_bofa_day_start_backtests
    GROUP BY day_of_week
    ORDER BY sessions DESC;
""").df()
display(dow_perf_df)
